<a href="https://colab.research.google.com/github/petrodyak/Assignment1/blob/main/Copy_of_%D0%97%D0%B0%D0%BD%D1%8F%D1%82%D1%82%D1%8F_6_%E2%80%93%C2%A0%D0%86%D0%BD%D1%81%D1%82%D1%80%D1%83%D0%BC%D0%B5%D0%BD%D1%82%D0%B8_%D0%BE%D1%80%D0%BA%D0%B5%D1%81%D1%82%D1%80%D0%B0%D1%86%D1%96%D1%97_AI_%D0%B0%D0%B3%D0%B5%D0%BD%D1%82%D1%96%D0%B2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🎓 Заняття 6. Інструменти оркестрації AI агентів

In [ ]:
!pip install -q "langgraph>=1.0" "langchain-openai>=0.4" "langchain-core>=0.3" "grandalf" "crewai>=1.0" "smolagents>=1.0" "openai-agents"

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.6/43.6 kB 2.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 68.6/68.6 kB 2.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.5/40.5 kB 2.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 4.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 68.1/68.1 kB 3.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 87.7/87.7 kB 6.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 41.8/41.8 kB 2.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 929.3/929.3 kB 11.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 155.7/155.7 kB 7.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 216.4/216.4 kB 8.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 19.9/19.9 MB 19.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 98.2/98.2 kB 7.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 15

In [ ]:
import os
import warnings
from getpass import getpass

warnings.filterwarnings("ignore")

if "OPENAI_API_KEY" not in os.environ:
    os.environ["OPENAI_API_KEY"] = getpass("Enter your OpenAI API key: ")

print("API key configured!")

Enter your OpenAI API key: ··········
API key configured!


## Проблема масштабування: чому одного агента мало?

Коли задача стає занадто складною, єдиний "всемогутній" промпт починає давати збої.

| Проблема | Що відбувається                                        |
|----------|--------------------------------------------------------|
| 🧠 **Контекстне вікно** | Агент губить деталі при багатьох кроках                |
| 🌀 **Галюцинації** | Один агент вигадує факти, і немає кому його перевірити |
| 🔄 **Нескінченні цикли** | Агент застрягає в повторенні однієї помилки            |
| 🎯 **Розмитий фокус** | Один агент із десятьма обов'язками робить кожен з них посередньо |


### Принцип розділення відповідальності (Separation of Concerns)

Той самий принцип, що працює в розробці ПЗ, працює і з агентами:

> **Чим простіша задача — тим легше з нею впоратися.**

Один агент, що одночасно досліджує, аналізує, пише і перевіряє — це як монолітний додаток на 10 000 рядків в одному файлі. Натомість, якщо кожен агент відповідає за **одну чітку функцію**, він:
- отримує **коротший і точніший промпт** → менше галюцинацій
- працює з **меншим контекстом** → не губить деталі
- легше **тестується і замінюється** → простіше відлагодження

**Рішення:** Розподіл праці — кілька спеціалізованих агентів, кожен відповідає за свою частину.

```
  Один агент (монолітний)           Команда агентів (розподілена)
  ┌──────────────────┐              ┌───────────┐  ┌──────────┐
  │  Досліджує       │              │ Дослідник │──│ Аналітик │
  │  Аналізує        │    ──►       └───────────┘  └──────────┘
  │  Пише            │                    │              │
  │  Перевіряє       │              ┌───────────┐  ┌──────────┐
  │  Деплоїть        │              │   Автор   │──│ Рев'юер  │
  └──────────────────┘              └───────────┘  └──────────┘
```

### Коли мультиагенти НЕ потрібні

Перш ніж будувати складну мультиагентну систему, варто переконатись, що вона дійсно потрібна. Мультиагентна архітектура додає overhead, який не завжди виправданий:

| Фактор         | Що додає мультиагентність |
|----------------|--------------------------|
| **Токени**     | Кожна передача контексту між агентами — це додаткові токени |
| **Затримка**   | Більше LLM-викликів = довше очікування |
| **Debugging**  | Помилка може бути в агенті, в оркестрації, або в передачі контексту між ними |
| **Складність** | Більше рухомих частин = більше точок відмови |

**Правило:** Якщо задачу можна вирішити одним чітким промптом з кількома tools — мультиагенти зайві.

**Мультиагенти корисні, коли:**
- Потрібні **різні моделі або промпти** для різних етапів (наприклад, дешева модель для класифікації + потужна для генерації)
- Є **ітеративна перевірка** (один пише, інший рецензує — цикл повторюється)
- Задача **природно ділиться на незалежні підзадачі**, які можна виконувати паралельно
- Потрібна **ізоляція контексту** — кожен агент бачить лише релевантну частину інформації

## Що таке оркестрація?

Оркестрація — це управління **командою агентів**: хто виконує задачу, в якому порядку, і як передається результат між ними.

```
                  ┌─────────────────────────┐
                  │       ОРКЕСТРАТОР       │
                  │   Маршрутизація · Стан  │
                  │ Помилки · Human-in-loop │
                  └────────────┬────────────┘
                 ╱             │             ╲
  ┌─────────────────┐  ┌────────────────┐  ┌────────────────┐
  │     Агент A     │  │     Агент B    │  │     Агент C    │
  │ ┌──────┬──────┐ │  │ ┌─────┬──────┐ │  │ ┌─────┬──────┐ │
  │ │ Tools│Memory│ │  │ │Tools│Memory│ │  │ │Tools│Memory│ │
  │ └──────┴──────┘ │  │ └─────┴──────┘ │  │ └─────┴──────┘ │
  │       LLM       │  │      LLM       │  │      LLM       │
  └─────────────────┘  └────────────────┘  └────────────────┘
```

- **Оркестратор** — вирішує, якому агенту делегувати задачу, контролює стан і обробляє помилки
- **Агент** — автономна одиниця зі своїм LLM, інструментами та пам'яттю
- Кожен агент відповідає за **свою спеціалізацію** і може працювати незалежно

## П'ять головних парадигм оркестрації

| Парадигма | Фреймворк | Метафора | Головна ідея |
|-----------|-----------|----------|-------------|
| 🔀 **Графова** | LangGraph | Блок-схема | Жорсткий контроль потоку через вузли та ребра |
| 👥 **Рольова** | CrewAI | Команда співробітників | Агенти з ролями, цілями та передісторією |
| 💬 **Розмовна** | Microsoft Agent Framework | Протокол обміну повідомленнями | Агенти спілкуються, поки задача не вирішена |
| 🐍 **Код-орієнтована** | Smolagents | Скрипт | Генерація коду замість JSON для виклику інструментів |
| 🤝 **Handoff** | OpenAI Agents SDK | Естафета | Агент передає управління іншому через виклик функції |

### 🔀 Графова парадигма
Розробник явно описує **граф виконання**: вузли (що робити) і ребра (куди далі). Оркестратор йде по графу крок за кроком, як процесор виконує блок-схему. Це дає максимальну передбачуваність — ви точно знаєте, який агент спрацює і за яких умов. Підходить для бізнес-процесів, де кожен крок має бути детермінованим.

### 👥 Рольова парадигма
Кожному агенту задається **роль**, **ціль** і **передісторія** — ніби ви наймаєте команду спеціалістів. Замість інженерії графів ви описуєте "хто є хто" і "хто що має зробити", а фреймворк організовує взаємодію. Низький поріг входження, але менше контролю над точними кроками виконання.

### 💬 Розмовна парадигма
Агенти поміщаються у спільну розмову і **обмінюються повідомленнями** по черзі. Стратегія вибору наступного спікера визначає порядок (round-robin, selector). Цикл продовжується, поки не виконається умова зупинки (наприклад, ключове слово "APPROVE"). Ідеально для задач, де потрібна ітеративна перевірка — один агент пише, інший рецензує.

### 🐍 Код-орієнтована парадигма
**CodeAgent** генерує Python-код замість JSON для виклику інструментів — це економніше за токенами і дозволяє комбінувати кілька інструментів в одному блоці коду. Мінімум абстракцій, максимум прозорості.

### 🤝 Handoff парадигма
Агент сам вирішує **передати управління** іншому агенту через виклик функції (наприклад, `handoffs=[billing, tech]`). Немає центрального оркестратора — агенти передають "естафету" один одному залежно від контексту запиту.

> 💡 Кожна парадигма має свої сильні сторони — "срібної кулі" не існує.

---
# 📚 LangGraph — Графова оркестрація

## Від ланцюжків до графів

LangChain створив LangGraph, бо лінійні ланцюжки (Chains) не підтримують **цикли**.

```
  Лінійний ланцюжок (Chain)        Граф із циклами (LangGraph)

  [Input] → [LLM] → [Output]      [Input] → [Agent] → [Tool]
                                     ↑         │
                                     │         ↓
                                     └───── [Check] ──→ [Output]
                                           (помилка?)
```

**Приклад:** Агент пише код → отримує помилку → переписує код → знову перевіряє.
Лінійний ланцюжок не може повернутися назад, а граф — може.

## Основні компоненти LangGraph

| Компонент | Опис | Аналогія |
|-----------|------|----------|
| **State** | `TypedDict` або Pydantic-модель, що зберігає весь контекст | Спільна дошка |
| **Nodes** | Python-функції, що отримують State, виконують роботу, повертають оновлення | Робочі станції |
| **Edges** | Правила переходу між вузлами (звичайні та умовні — `conditional_edges`) | Стрілки на блок-схемі |
| **START / END** | Спеціальні вузли для початку та завершення графа | Вхід / Вихід |

Node **не перезаписує весь State** — він повертає лише ті поля, які хоче оновити. LangGraph зливає ці оновлення з поточним станом.

## Checkpoints та Human-in-the-Loop

Одна з найсильніших сторін LangGraph — **вбудована персистентність**.

**Human-in-the-Loop (HITL)** — патерн, коли система ставиться на паузу і чекає рішення людини перед виконанням критичної дії. Наприклад, агент підготував email — але перед відправкою людина має підтвердити текст. Це дозволяє використовувати автономність агентів, зберігаючи контроль над незворотними діями.

### Що дає Checkpointer?

| Можливість | Опис |
|------------|------|
| ⏸️ **Пауза** | Зупинити виконання графа для отримання дозволу від людини |
| ▶️ **Відновлення** | Продовжити з тієї ж точки після підтвердження |
| ⏪ **Time Travel** | "Відмотати" стан назад до будь-якого попереднього кроку |
| 💾 **Збереження** | Стан зберігається між сесіями (SQLite, PostgreSQL, Redis) |

## Типові Enterprise-патерни

### Customer Support Routing

Найпоширеніший патерн — класифікація запиту і маршрутизація до спеціаліста:

| Крок | Вузол | Дія |
|------|-------|-----|
| 1 | **Користувач** | Надсилає запит: "My internet is down" |
| 2 | **Класифікатор** | LLM визначає категорію → `"technical"` |
| 3 | **conditional_edge** | Читає `category` → направляє до `technical_agent` |
| 4 | **Technical Agent** | Генерує відповідь: "Try restarting your router..." |
| 5 | **END** | Або ескалація на людину, якщо агент не впорався |

Потік: **Користувач → Класифікатор → (Billing / Technical / General) → Відповідь / Ескалація**

Цей патерн реалізовано в Прикладі 1 нижче.

### Інші патерни

| Патерн | Ідея | Коли використовувати |
|--------|------|---------------------|
| **Multi-Agent Collaboration** | Кілька агентів працюють над спільною задачею по черзі | Дослідження + написання звіту |
| **Supervisor Agent** | Один "менеджер" делегує підзадачі спеціалістам | Складний запит із кількома аспектами |
| **Plan-and-Execute** | Спочатку створити план, потім виконувати крок за кроком | Багатокрокові задачі з невідомою кількістю кроків |

## Плюси та мінуси LangGraph

| ✅ Плюси                        | ❌ Мінуси |
|--------------------------------|----------|
| Тотальний контроль над потоком | Крута крива навчання |
| Підходить для production       | Багатослівний (boilerplate) код |
| Надійне збереження стану       | Складно візуалізувати великі графи без LangSmith |
| Потоковий вивід (streaming)    | Потрібно глибоке розуміння графової моделі |
| Широка екосистема LangChain    | Overhead для простих задач |

### 💻 Приклад 1: Маршрутизація запитів з LLM

Реальний Customer Support Router — LLM класифікує запит і направляє до спеціалізованого агента.

In [ ]:
from typing import TypedDict, Annotated, Literal
from operator import add
from langgraph.graph import StateGraph, START, END
from langgraph.checkpoint.memory import MemorySaver
from langchain_openai import ChatOpenAI
from langchain_core.messages import HumanMessage, AIMessage, SystemMessage


llm = ChatOpenAI(model="gpt-5.4-nano", temperature=0)


class SupportState(TypedDict):
    messages: Annotated[list, add]
    category: str


def classify(state: SupportState) -> dict:
    response = llm.invoke([
        SystemMessage(content=
            "Classify the customer request into one of the categories: billing, technical, general.\n"
            "Respond with ONLY one word: billing, technical, or general."
        ),
        state["messages"][-1]
    ])
    category = response.content.strip().lower()
    if category not in ("billing", "technical", "general"):
        category = "general"
    print(f"  Classified as: {category}")
    return {"category": category}


def billing_agent(state: SupportState) -> dict:
    response = llm.invoke([
        SystemMessage(content="You are a billing specialist. Help with payments, invoices, and plans. Be concise."),
        *state["messages"]
    ])
    return {"messages": [AIMessage(content=response.content)]}

def technical_agent(state: SupportState) -> dict:
    response = llm.invoke([
        SystemMessage(content="You are a technical specialist. Resolve issues with internet, equipment, and settings. Be concise."),
        *state["messages"]
    ])
    return {"messages": [AIMessage(content=response.content)]}

def general_agent(state: SupportState) -> dict:
    response = llm.invoke([
        SystemMessage(content="You are a general support operator. Answer general questions. Be concise."),
        *state["messages"]
    ])
    return {"messages": [AIMessage(content=response.content)]}


graph = StateGraph(SupportState)
graph.add_node("classify", classify)
graph.add_node("billing", billing_agent)
graph.add_node("technical", technical_agent)
graph.add_node("general", general_agent)

graph.add_edge(START, "classify")

graph.add_conditional_edges(
    "classify",
    lambda state: state["category"],
    {"billing": "billing", "technical": "technical", "general": "general"}
)
graph.add_edge("billing", END)
graph.add_edge("technical", END)
graph.add_edge("general", END)

memory = MemorySaver()
support_app = graph.compile(checkpointer=memory)

print("Customer Support Router built!")
print(support_app.get_graph().draw_ascii())

Customer Support Router built!
                      +-----------+                       
                      | __start__ |                       
                      +-----------+                       
                            *                             
                            *                             
                            *                             
                      +----------+                        
                      | classify |                        
                   ...+----------+....                    
               ....         .         ....                
           ....             .             ....            
         ..                 .                 ..          
+---------+           +---------+           +-----------+ 
| billing |           | general |           | technical | 
+---------+****       +---------+         **+-----------+ 
               ****         *         ****                
                   ****  

In [ ]:
# Test with different queries
test_queries = [
    "Why is my February bill so high?",
    "Wi-Fi keeps disconnecting, help!",
    "What are your office hours?"
]

for i, query in enumerate(test_queries):
    print("\n" + "=" * 60)
    print(f"Customer: {query}")
    result = support_app.invoke(
        {"messages": [HumanMessage(content=query)], "category": ""},
        config={"configurable": {"thread_id": f"test-{i}"}}
    )
    print(f"Response: {result['messages'][-1].content}")


Customer: Why is my February bill so high?
  Classified as: billing
Response: I can help—usually a “high” February bill comes from one (or more) of these:

- **Plan/price change**: moved to a higher tier or add-ons were enabled.
- **Proration**: mid-cycle upgrade/downgrade can charge partial amounts.
- **Usage overages**: extra usage (e.g., data, seats, API calls, storage, minutes).
- **Billing cycle difference**: February has fewer days, so sometimes the bill reflects a **catch-up** or **true-up** from the previous period.
- **One-time charges**: setup fees, hardware/annual fees, professional services, credits that were removed, etc.
- **Multiple invoices**: two billing periods or subscriptions posting in February.

To pinpoint it quickly, can you share (paste text is fine):
1) Your **invoice amount** and **invoice date**  
2) The **line items** from the February invoice (or a screenshot text)  
3) Whether you changed plan around late January/early February (yes/no)

Once I see the l

In [ ]:
# Checkpointer demo: state is persisted between invocations
config = {"configurable": {"thread_id": "test-0"}}
state = support_app.get_state(config)
print(f"Saved state for thread test-0:")
print(f"  Category: {state.values['category']}")
print(f"  Message count: {len(state.values['messages'])}")
print(f"  Last message: {state.values['messages'][-1].content[:100]}...")

# Time Travel: review all steps
print("\nTime Travel (step history):")
for snapshot in support_app.get_state_history(config):
    step = snapshot.metadata.get("step", "?")
    node = snapshot.metadata.get("source", "?")
    cat = snapshot.values.get("category", "—")
    n_msgs = len(snapshot.values.get("messages", []))
    print(f"  Step {step} ({node}): category={cat}, messages={n_msgs}")

Saved state for thread test-0:
  Category: billing
  Message count: 2
  Last message: I can help—usually a “high” February bill comes from one (or more) of these:

- **Plan/price change*...

Time Travel (step history):
  Step 2 (loop): category=billing, messages=2
  Step 1 (loop): category=billing, messages=1
  Step 0 (loop): category=, messages=1
  Step -1 (input): category=—, messages=0


### Що щойно відбулось: потік даних крок за кроком

Розберемо, як State змінювався під час виконання запиту `"Why is my February bill so high?"`:

---

**Крок 1: START** — граф отримує початковий State

| Поле | Значення |
|------|----------|
| `messages` | `[HumanMessage("Why is my February bill so high?")]` |
| `category` | `""` |

↓ *edge* → `classify`

---

**Крок 2: Node `classify`** — LLM визначає категорію

Node повертає **лише одне поле**: `{"category": "billing"}`

| Поле | Значення | Що сталося |
|------|----------|------------|
| `messages` | `[HumanMessage("Why is my February bill so high?")]` | не змінено — node не повернув це поле |
| `category` | `"billing"` | оновлено |

↓ *conditional_edge:* `lambda state: state["category"]` → `"billing"` → граф іде до вузла `billing`

---

**Крок 3: Node `billing_agent`** — спеціалізований агент генерує відповідь

Node повертає: `{"messages": [AIMessage("Your February bill increased because...")]}`

| Поле | Значення | Що сталося |
|------|----------|------------|
| `messages` | `[HumanMessage("Why is my February bill..."), AIMessage("Your February bill...")]` | новий елемент **додано до списку** |
| `category` | `"billing"` | не змінено |

↓ *edge* → **END**

---

**Зверніть увагу:** `billing_agent` повернув `{"messages": [AIMessage(...)]}` — але стара `HumanMessage` не зникла! Новий елемент **додався** до існуючого списку, а не замінив його. Це працює завдяки **reducer-функції**:

```python
messages: Annotated[list, add]  # operator.add — нові елементи додаються до списку
category: str                   # звичайне поле — просто перезаписується
```

Без `Annotated[list, add]` кожне оновлення `messages` затирало б попередні повідомлення, і до кінця графа залишалась би лише остання відповідь.

### 💻 Приклад 2: ReAct Agent з інструментами

LangGraph має вбудований `create_react_agent` — повноцінний агент із циклом **Thought → Action → Observation**.

In [ ]:
from langgraph.prebuilt import create_react_agent
from langchain_core.tools import tool
import ast
import operator

@tool
def get_exchange_rate(base: str, target: str) -> str:
    """Returns the exchange rate. base and target are currency codes (USD, EUR, UAH, GBP)."""
    rates = {
        ("USD", "UAH"): 41.50, ("EUR", "UAH"): 45.20,
        ("GBP", "UAH"): 52.80, ("USD", "EUR"): 0.92,
    }
    rate = rates.get((base, target))
    if rate:
        return f"1 {base} = {rate} {target}"
    reverse = rates.get((target, base))
    if reverse:
        return f"1 {base} = {1/reverse:.4f} {target}"
    return f"Rate {base}/{target} not found"

# Safe math operations whitelist
_SAFE_OPS = {
    ast.Add: operator.add,
    ast.Sub: operator.sub,
    ast.Mult: operator.mul,
    ast.Div: operator.truediv,
    ast.Pow: operator.pow,
    ast.USub: operator.neg,
}

def _safe_eval(node):
    """Recursively evaluate an AST node using only whitelisted arithmetic operations."""
    if isinstance(node, ast.Expression):
        return _safe_eval(node.body)
    if isinstance(node, ast.Constant) and isinstance(node.value, (int, float)):
        return node.value
    if isinstance(node, ast.BinOp) and type(node.op) in _SAFE_OPS:
        return _SAFE_OPS[type(node.op)](_safe_eval(node.left), _safe_eval(node.right))
    if isinstance(node, ast.UnaryOp) and type(node.op) in _SAFE_OPS:
        return _SAFE_OPS[type(node.op)](_safe_eval(node.operand))
    raise ValueError(f"Unsupported expression: {ast.dump(node)}")

@tool
def calculate(expression: str) -> str:
    """Evaluates a math expression. For example: '100 * 41.50' or '(500 + 300) / 2'."""
    try:
        tree = ast.parse(expression, mode="eval")
        result = _safe_eval(tree)
        return f"{expression} = {result}"
    except Exception as e:
        return f"Calculation error: {e}"

# Create a ReAct Agent
react_agent = create_react_agent(
    model=ChatOpenAI(model="gpt-5.4-nano", temperature=0),
    tools=[get_exchange_rate, calculate],
    prompt="You are a financial assistant. Use tools for precise calculations."
)

print("ReAct Agent created!")
print(f"  Tools: {[t.name for t in [get_exchange_rate, calculate]]}")

ReAct Agent created!
  Tools: ['get_exchange_rate', 'calculate']


In [ ]:
# Run — the agent autonomously decides which tools to call
result = react_agent.invoke(
    {"messages": [HumanMessage(content="How many UAH is 250 USD? And 100 EUR?")]}
)

# Show the full reasoning chain
for msg in result["messages"]:
    role = msg.__class__.__name__.replace("Message", "")
    content = getattr(msg, "content", "")
    if content:
        print(f"\n[{role}] {content[:300]}")
    # Show tool calls
    if hasattr(msg, "tool_calls") and msg.tool_calls:
        for tc in msg.tool_calls:
            print(f"  Tool call: {tc['name']}({tc['args']})")


[Human] How many UAH is 250 USD? And 100 EUR?
  Tool call: get_exchange_rate({'base': 'USD', 'target': 'UAH'})
  Tool call: get_exchange_rate({'base': 'EUR', 'target': 'UAH'})

[Tool] 1 USD = 41.5 UAH

[Tool] 1 EUR = 45.2 UAH
  Tool call: calculate({'expression': '250 * 41.5'})
  Tool call: calculate({'expression': '100 * 45.2'})

[Tool] 250 * 41.5 = 10375.0

[Tool] 100 * 45.2 = 4520.0

[AI] - **250 USD** ≈ **10,375 UAH** (at 1 USD = 41.5 UAH)  
- **100 EUR** ≈ **4,520 UAH** (at 1 EUR = 45.2 UAH)


---
# 🔧 CrewAI — Рольова співпраця

## Філософія CrewAI

CrewAI пропонує думати про агентів **не як про функції, а як про співробітників**.

Замість інженерії графів — "менеджмент команди":

- Кожен агент має **роль** (role), **ціль** (goal) та **передісторію** (backstory)
- Передісторія формує системний промпт, що визначає стиль роботи агента
- Фреймворк створений для **швидкого прототипування** та **генерації контенту**

```
  ┌─────────────────────────────────────────┐
  │              CREW (Команда)             │
  │                                         │
  │  👨‍🔬 Senior Researcher                   │
  │    role: "Дослідник ринку"              │
  │    goal: "Знайти ключові тренди"        │
  │    backstory: "10 років у Wall Street"  │
  │                                         │
  │  ✍️ Copywriter                           │
  │    role: "Автор контенту"               │
  │    goal: "Написати звіт для клієнта"    │
  │    backstory: "Фінансовий журналіст"    │
  └─────────────────────────────────────────┘
```

## Анатомія CrewAI: Agent, Task, Crew

| Компонент | Ключові параметри | Призначення |
|-----------|-------------------|------------|
| **Agent** | `role`, `goal`, `backstory`, `tools` | Спеціалізований виконавець із характером |
| **Task** | `description`, `expected_output`, `agent` | Конкретне завдання з очікуваним результатом |
| **Crew** | `agents`, `tasks`, `process` | Контейнер, що об'єднує агентів і задачі |

**Принцип:** Task привʼязується до конкретного Agent. Crew запускає виконання.

## Організація процесів (Process Topology)

### Sequential (Послідовний)
```
   Agent A         Agent B         Agent C
  ┌──────┐        ┌──────┐        ┌──────┐
  │Task 1│───────▶│Task 2│───────▶│Task 3│───▶ Результат
  └──────┘        └──────┘        └──────┘
```
Агенти виконують задачі по черзі, передаючи результат наступному.

### Hierarchical (Ієрархічний)
```
             ┌───────────────┐
             │   Менеджер    │ (створюється автоматично)
             │  (Supervisor) │
             └──────┬────────┘
               ╱    │    ╲
       ┌───────┐┌───────┐┌───────┐
       │Agent A││Agent B││Agent C│
       └───────┘└───────┘└───────┘
```
Менеджер сам вирішує, кому делегувати частини завдання.

### Як агенти насправді "спілкуються"?

CrewAI агенти **не мають спільної пам'яті і не обмінюються повідомленнями напряму**. Комунікація відбувається через передачу тексту між задачами:

1. **Sequential**: Agent A виконує Task 1 → текстовий результат автоматично вставляється в промпт Task 2 як контекст → Agent B працює з ним. Кожен агент бачить лише вхід своєї задачі + результат попередньої.

2. **Hierarchical**: CrewAI створює прихованого "manager agent", який отримує всі задачі. Він вирішує, якому агенту делегувати кожну підзадачу, збирає результати і формує фінальну відповідь. Делегування реалізовано як tool call — менеджер викликає "Ask Agent X" як інструмент.

3. **Delegation** (`allow_delegation=True`): Агент отримує додатковий tool "Delegate work to coworker", який дозволяє йому надіслати запит іншому агенту в тій самій Crew. Під капотом — це LLM-виклик до іншого агента з переданим контекстом.

**Тобто:** Ніякого протоколу обміну повідомленнями немає. Є просто ланцюжок LLM-викликів, де вихід одного стає частиною промпта іншого.

## Інструменти та Делегування

### Інструменти (Tools)
Агенти можуть використовувати:
- 🔍 Пошук в інтернеті (SerperDevTool, ScrapeWebsiteTool)
- 📄 Читання та аналіз файлів
- 🛠️ Кастомні Python-функції

### Делегування (Delegation)
Унікальна можливість: якщо агент **не знає відповіді**, він може запитати іншого агента у своїй команді.

```
  Дослідник: "Яка вартість акцій?"
       │
       ▼  (делегує)
  Аналітик: "Поточна ціна: $185.20"
       │
       ▼  (повертає результат)
  Дослідник: "Ціна $185.20 — нижче за середню за квартал."
```

## Плюси та мінуси CrewAI

| ✅ Плюси | ❌ Мінуси |
|----------|----------|
| Низький поріг входження | "Чорний ящик" під капотом |
| Дуже читабельний код (майже як текст) | Важко контролювати точні кроки агента |
| Ідеально для творчих задач | Менша придатність для суворих бізнес-процесів |
| Вбудоване делегування між агентами | Залежність від якості промптів |

### 💻 Приклад коду CrewAI

In [ ]:
from crewai import Agent, Task, Crew, Process

# Real scenario: competitive analysis and report writing

researcher = Agent(
    role="Competitive Analyst",
    goal="Compare LangGraph, CrewAI, and Microsoft Agent Framework by key criteria",
    backstory=(
        "You are a technical analyst with AI/ML experience. "
        "You objectively evaluate technologies, highlighting pros and cons."
    ),
    verbose=True,
    allow_delegation=False
)

writer = Agent(
    role="Technical Copywriter",
    goal="Turn technical analysis into a clear recommendation for a CTO",
    backstory=(
        "You are an experienced technical writer. "
        "You write concisely, with structure, and always give a specific recommendation."
    ),
    verbose=True,
    allow_delegation=False
)

research_task = Task(
    description=(
        "Compare LangGraph, CrewAI, and Microsoft Agent Framework by criteria: "
        "1) ease of use, 2) flow control, "
        "3) production readiness. "
        "For each framework list 2 pros and 2 cons."
    ),
    expected_output="Comparison table with pros and cons for each framework.",
    agent=researcher
)

report_task = Task(
    description=(
        "Based on the research, write a short recommendation (up to 200 words) "
        "for a startup CTO building a customer support system. "
        "Specify which framework to choose and why."
    ),
    expected_output="Recommendation up to 200 words with justification.",
    agent=writer
)

crew = Crew(
    agents=[researcher, writer],
    tasks=[research_task, report_task],
    process=Process.sequential,
    verbose=True
)

print(f"Crew created: {[a.role for a in crew.agents]}")

Crew created: ['Competitive Analyst', 'Technical Copywriter']


In [ ]:
# Run CrewAI — real LLM execution
result = crew.kickoff()

print("=" * 60)
print("FINAL CREW RESULT:")
print("=" * 60)
print(result.raw)

╭─────────────────────────────────────────── 🚀 Crew Execution Started ───────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Started                                                                                         │
│  Name: crew                                                                                                     │
│  ID: 63c13887-af01-4b3e-8ef6-89170584164a                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── 📋 Task Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Started                                                                                                   │
│  Name: Compare LangGraph, CrewAI, and Microsoft Agent Framework by criteria: 1) ease of use, 2) flow control,   │
│  3) production readiness. For each framework list 2 pros and 2 cons.                                            │
│  ID: c96947ea-28de-40f9-b537-a2c468aa86c4                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Competitive Analyst                                                                                     │
│                                                                                                                 │
│  Task: Compare LangGraph, CrewAI, and Microsoft Agent Framework by criteria: 1) ease of use, 2) flow control,   │
│  3) production readiness. For each framework list 2 pros and 2 cons.                                            │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭───────────────────────────────────────────── ✅ Agent Final Answer ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Competitive Analyst                                                                                     │
│                                                                                                                 │
│  Final Answer:                                                                                                  │
│  | Criteria            | LangGraph                              | CrewAI                                |       │
│  Microsoft Agent Framework             |                                                                        │
│  |---------------------|--------------------------------------|-------------------------------------|---------  │
│  ----------------------------|                                                                                  │
│  | **Ease of Use**      | **Pros:**                            | **Pros:**                           |          │
│  **Pros:**                           |                                                                          │
│  |                     | 1. Intuitive visual interface that simplifies building complex workflows. | 1.         │
│  Streamlined APIs designed for rapid prototyping. | 1. Well-documented with extensive Microsoft ecosystem       │
│  support. |                                                                                                     │
│  |                     | 2. Low-code environment accessible to users without deep ML knowledge. | 2. Modular    │
│  design allows easy component reuse. | 2. Integrates seamlessly with Azure services, facilitating scalability.  │
│  |                                                                                                              │
│  |                     | **Cons:**                            | **Cons:**                          | **Cons:**  │
│  |                                                                                                              │
│  |                     | 1. Documentation can be sparse for advanced customization. | 1. Steeper learning       │
│  curve for advanced features. | 1. Requires familiarity with Microsoft stack, which may be cumbersome for some  │
│  users. |                                                                                                       │
│  |                     | 2. Limited community support and third-party tutorials. | 2. Less beginner-friendly    │
│  than some competitors. | 2. UI and tooling sometimes feel outdated compared to newer frameworks. |             │
│  | **Flow Control**     | **Pros:**                            | **Pros:**                          |           │
│  **Pros:**                          |                                                                           │
│  |                     | 1. Supports declarative graph-based flow control enabling complex path management. |   │
│  1. Provides dynamic flow control with real-time decision making. | 1. Robust state management and              │
│  event-driven architecture. |                                                                                   │
│  |                     | 2. Easy to visualize and debug flow paths. | 2. Supports multi-agent coordination and  │
│  conflict resolution strategies. | 2. Supports parallel and conditional task executions effectively. |          │
│  |                     | **Cons:**                            | **Cons:**                         | **Cons:**   │
│  |                                                                                                              │
│  |                     | 1. Limited flexibility with asynchronous event handling in some cases. | 1. Can        │
│  become unwieldy for very large workflows without caref

╭────────────────────────────────────────────── 📋 Task Completion ───────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Completed                                                                                                 │
│  Name: Compare LangGraph, CrewAI, and Microsoft Agent Framework by criteria: 1) ease of use, 2) flow control,   │
│  3) production readiness. For each framework list 2 pros and 2 cons.                                            │
│  Agent: Competitive Analyst                                                                                     │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── 📋 Task Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Started                                                                                                   │
│  Name: Based on the research, write a short recommendation (up to 200 words) for a startup CTO building a       │
│  customer support system. Specify which framework to choose and why.                                            │
│  ID: 9f1e1dcd-e30b-45a3-a620-9ff38d1baa80                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Technical Copywriter                                                                                    │
│                                                                                                                 │
│  Task: Based on the research, write a short recommendation (up to 200 words) for a startup CTO building a       │
│  customer support system. Specify which framework to choose and why.                                            │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭───────────────────────────────────────────── ✅ Agent Final Answer ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Technical Copywriter                                                                                    │
│                                                                                                                 │
│  Final Answer:                                                                                                  │
│  For a startup CTO building a customer support system, I recommend choosing **LangGraph** as the foundational   │
│  framework. LangGraph offers an intuitive, low-code, visual interface that drastically reduces development      │
│  time and allows your team to rapidly prototype and iterate complex workflows without deep machine learning     │
│  expertise—crucial for fast-moving startups. Its graph-based flow control simplifies designing and debugging    │
│  the customer interaction paths, providing clear transparency essential for maintaining quality support         │
│  experiences.                                                                                                   │
│                                                                                                                 │
│  While LangGraph is newer and has a smaller community, its scalable infrastructure and deployment pipeline      │
│  readiness align well with cloud-native growth strategies. Unlike CrewAI, which introduces a steeper learning   │
│  curve and evolving features, LangGraph strikes a better balance between ease of use and production readiness.  │
│  Compared to Microsoft Agent Framework, LangGraph avoids the high licensing costs and complexity that can       │
│  burden early-stage teams, especially if your stack isn’t Microsoft-centric.                                    │
│                                                                                                                 │
│  To mitigate its limitations, invest early in internal documentation and consider engaging with the LangGraph   │
│  community to accelerate your team’s proficiency. This approach delivers a robust, scalable, and                │
│  developer-friendly base for your customer support system, enabling both speed and control as your startup      │
│  scales.                                                                                                        │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭────────────────────────────────────────────── 📋 Task Completion ───────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Completed                                                                                                 │
│  Name: Based on the research, write a short recommendation (up to 200 words) for a startup CTO building a       │
│  customer support system. Specify which framework to choose and why.                                            │
│  Agent: Technical Copywriter                                                                                    │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

FINAL CREW RESULT:
For a startup CTO building a customer support system, I recommend choosing **LangGraph** as the foundational framework. LangGraph offers an intuitive, low-code, visual interface that drastically reduces development time and allows your team to rapidly prototype and iterate complex workflows without deep machine learning expertise—crucial for fast-moving startups. Its graph-based flow control simplifies designing and debugging the customer interaction paths, providing clear transparency essential for maintaining quality support experiences.

While LangGraph is newer and has a smaller community, its scalable infrastructure and deployment pipeline readiness align well with cloud-native growth strategies. Unlike CrewAI, which introduces a steeper learning curve and evolving features, LangGraph strikes a better balance between ease of use and production readiness. Compared to Microsoft Agent Framework, LangGraph avoids the high licensing costs and complexity that can burd

### 🛠️ CrewAI: Агент з кастомним інструментом

In [ ]:
from crewai.tools import tool

@tool
def knowledge_base_search(query: str) -> str:
    """Searches the company's internal knowledge base. Takes a search query."""
    # Simulated knowledge base
    kb = {
        "plans": "Basic: $5/mo, Standard: $9/mo, Premium: $14/mo",
        "setup": "Setup is free. Timeline: 1-3 business days.",
        "contacts": "Phone: 1-800-123-456. Email: support@example.com",
        "wi-fi": "Recommended router: TP-Link Archer AX55. Setup: 192.168.0.1"
    }
    for key, value in kb.items():
        if key in query.lower():
            return value
    return f"No results found for '{query}'."

support_agent = Agent(
    role="Support Operator",
    goal="Answer customer questions using the knowledge base",
    backstory="You are a call center operator with 3 years of experience.",
    tools=[knowledge_base_search],
    verbose=True
)

support_task = Task(
    description="Customer asks: 'What are your plans and how much is setup?'. Answer in English.",
    expected_output="Full answer with plan info and setup details.",
    agent=support_agent
)

support_crew = Crew(agents=[support_agent], tasks=[support_task], verbose=True)
result = support_crew.kickoff()
print(f"\nAnswer: {result.raw}")

╭─────────────────────────────────────────────── Execution Traces ────────────────────────────────────────────────╮
│                                                                                                                 │
│  🔍 Detailed execution traces are available!                                                                    │
│                                                                                                                 │
│  View insights including:                                                                                       │
│    • Agent decision-making process                                                                              │
│    • Task execution flow and timing                                                                             │
│    • Tool usage details                                                                                         │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

Would you like to view your execution traces? [y/N] (20s timeout): 

╭─────────────────────────────────────────── 🚀 Crew Execution Started ───────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Started                                                                                         │
│  Name: crew                                                                                                     │
│  ID: 2181d99e-3408-46a4-93af-45601931df11                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── 📋 Task Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Started                                                                                                   │
│  Name: Customer asks: 'What are your plans and how much is setup?'. Answer in English.                          │
│  ID: e56427c3-c283-4ee0-8dfa-42b0ebb129cb                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Support Operator                                                                                        │
│                                                                                                                 │
│  Task: Customer asks: 'What are your plans and how much is setup?'. Answer in English.                          │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

...

╭──────────────────────────────────────────────── Crew Completion ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Completed                                                                                       │
│  Name: crew                                                                                                     │
│  ID: 63c13887-af01-4b3e-8ef6-89170584164a                                                                       │
│  Final Output: For a startup CTO building a customer support system, I recommend choosing **LangGraph** as the  │
│  foundational framework. LangGraph offers an intuitive, low-code, visual interface that drastically reduces     │
│  development time and allows your team to rapidly prototype and iterate complex workflows without deep machine  │
│  learning expertise—crucial for fast-moving startups. Its graph-based flow control simplifies designing and     │
│  debugging the customer interaction paths, providing clear transparency essential for maintaining quality       │
│  support experiences.                                                                                           │
│                                                                                                                 │
│  While LangGraph is newer and has a smaller community, its scalable infrastructure and deployment pipeline      │
│  readiness align well with cloud-native growth strategies. Unlike CrewAI, which introduces a steeper learning   │
│  curve and evolving features, LangGraph strikes a better balance between ease of use and production readiness.  │
│  Compared to Microsoft Agent Framework, LangGraph avoids the high licensing costs and complexity that can       │
│  burden early-stage teams, especially if your stack isn’t Microsoft-centric.                                    │
│                                                                                                                 │
│  To mitigate its limitations, invest early in internal documentation and consider engaging with the LangGraph   │
│  community to accelerate your team’s proficiency. This approach delivers a robust, scalable, and                │
│  developer-friendly base for your customer support system, enabling both speed and control as your startup      │
│  scales.                                                                                                        │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── Tracing Status ─────────────────────────────────────────────────╮
│                                                                                                                 │
│  Info: Tracing is disabled.                                                                                     │
│                                                                                                                 │
│  To enable tracing, do any one of these:                                                                        │
│  • Set tracing=True in your Crew/Flow code                                                                      │
│  • Set CREWAI_TRACING_ENABLED=true in your project's .env file                                                  │
│  • Run: crewai traces enable                                                                                    │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯


Answer: Our plans are as follows:
- Basic plan: $5 per month
- Standard plan: $9 per month
- Premium plan: $14 per month

Regarding the setup fee, there is no specific mention of any setup charge, so it appears there is no setup fee. If you need more details or have other questions, feel free to ask!


╭─────────────────────────────────────────────── Execution Traces ────────────────────────────────────────────────╮
│                                                                                                                 │
│  🔍 Detailed execution traces are available!                                                                    │
│                                                                                                                 │
│  View insights including:                                                                                       │
│    • Agent decision-making process                                                                              │
│    • Task execution flow and timing                                                                             │
│    • Tool usage details                                                                                         │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

Would you like to view your execution traces? [y/N] (20s timeout): 

---
# 🔬 Microsoft Agent Framework — Розмовна оркестрація

## Концепція Microsoft Agent Framework

Microsoft Agent Framework (MAF) — наступник AutoGen та Semantic Kernel, який об'єднує ідеї обох проєктів в єдину платформу. Обидва попередники перейшли у режим підтримки (maintenance mode). Еволюція: AutoGen 0.2 → AutoGen 0.4 (async redesign) → Microsoft Agent Framework (жовтень 2025, public preview).

> **Примітка:** AutoGen продовжує розвиватися як **AG2** (open-source проєкт під незалежною спільнотою AG2AI), а MAF — окремий продукт Microsoft. Якщо ви зустрінете AG2 — це не те саме, що MAF.

MAF використовує **багатоагентні чати** як основний механізм оркестрації:
1. Розробник створює агентів
2. Поміщає їх у **GroupChat** через `GroupChatBuilder`
3. Агенти обмінюються повідомленнями, поки не спрацює умова зупинки (termination condition)

```
  ┌─────────────────────────────────────────┐
  │          GROUP CHAT (Розмова)           │
  │                                         │
  │  🤖 Coder: "Ось код для Stack класу..." │
  │  🔍 Reviewer: "Не вистачає edge case!"  │
  │  🤖 Coder: "Додав обробку. Перевір."    │
  │  🔍 Reviewer: "Все чисто. APPROVE"      │
  │  🛑 termination_condition → СТОП        │
  └─────────────────────────────────────────┘
```

## Ключові компоненти Microsoft Agent Framework

| Компонент | Роль | Опис |
|-----------|------|------|
| **Agent** | 🤖 AI-агент | Агент з інструкціями, описом та інструментами |
| **AzureOpenAIResponsesClient** | 🔌 Модель | Клієнт для Azure OpenAI (наразі основний підтримуваний провайдер) |
| **GroupChatBuilder** | 🔄 Груповий чат | Конструктор групового чату з учасниками та умовами зупинки |
| **Termination functions** | 🛑 Умови зупинки | Python-функції, що перевіряють, чи час зупинити розмову |

### Як агенти координуються в GroupChat?

На відміну від CrewAI, де агенти передають текст по ланцюжку, в MAF агенти **бачать спільну історію повідомлень** — як учасники групового чату:

1. **Спільна пам'ять**: Усі повідомлення зберігаються в одному списку `conversation`. Кожен агент при своєму ході бачить **всю історію** — хто що сказав до нього.

2. **Хто говорить наступним?** GroupChat має стратегію вибору спікера:
   - **Round-robin** — агенти говорять по черзі в фіксованому порядку
   - **Selector** (за замовчуванням) — LLM аналізує контекст розмови і вирішує, який агент має відповісти наступним, на основі `description` кожного агента

3. **Коли зупинитись?** `termination_condition` — Python-функція, яка перевіряє останнє повідомлення після кожного ходу. Наприклад, `"APPROVE" in message` або `len(conversation) >= 10`.

**Ключова різниця з CrewAI:** В CrewAI задачі йдуть по черзі з фіксованим планом. В MAF розмова може розвиватися непередбачувано — Reviewer може попросити Coder переробити код кілька разів, і кількість ітерацій не визначена заздалегідь.

## Плюси та мінуси Microsoft Agent Framework

| ✅ Плюси | ❌ Мінуси |
|----------|----------|
| Найкращий для автоматичного написання та тестування коду | Величезне споживання токенів |
| Високий ступінь автономності | Діалог може зациклитися на одній помилці |
| Інтеграція з Azure та Semantic Kernel | Складніший debugging через чатову природу |
| Потужний GroupChat з різними стратегіями | Потребує Docker для безпечного виконання коду |
| Асинхронний API з першого дня | Прив'язка до екосистеми Microsoft |

### 💻 Приклад коду Microsoft Agent Framework

⚠️ **Наступні приклади MAF потребують Azure credentials та пакетів `agent-framework`, `azure-identity`.** Azure доступу немає, тому код наведено **лише для ознайомлення з API** — запускати його не потрібно.

Для встановлення пакетів (якщо маєте Azure доступ):
```bash
pip install "agent-framework>=1.0rc1" "azure-identity"
```

```python
# ⚠️ REFERENCE ONLY — requires Azure credentials

# Azure credentials — required only for MAF examples
from getpass import getpass

if "AZURE_AI_PROJECT_ENDPOINT" not in os.environ:
    os.environ["AZURE_AI_PROJECT_ENDPOINT"] = getpass("Enter Azure AI Project endpoint: ")
if "AZURE_AI_MODEL_DEPLOYMENT_NAME" not in os.environ:
    os.environ["AZURE_AI_MODEL_DEPLOYMENT_NAME"] = getpass("Enter Azure model deployment name: ")

from azure.identity import AzureCliCredential

print("Azure credentials configured!")
```

```python
# ⚠️ REFERENCE ONLY — requires Azure credentials

from agent_framework import Agent as MAFAgent
from agent_framework.azure import AzureOpenAIResponsesClient
from agent_framework.orchestrations import GroupChatBuilder

client = AzureOpenAIResponsesClient(
    project_endpoint=os.environ["AZURE_AI_PROJECT_ENDPOINT"],
    deployment_name=os.environ["AZURE_AI_MODEL_DEPLOYMENT_NAME"],
    credential=AzureCliCredential(),
)

# Real scenario: Coder writes code, Reviewer checks it
coder = MAFAgent(
    client=client,
    name="Coder",
    description="Writes clean Python code",
    instructions="You are a Python developer. Write clean, working code. "
                 "Always add usage examples and docstrings."
)

reviewer = MAFAgent(
    client=client,
    name="Reviewer",
    description="Reviews code for quality",
    instructions="You are an experienced code reviewer. Check for:\n"
                 "1) Logic correctness\n2) Edge cases\n3) Readability\n"
                 "If code is ready — respond: APPROVE"
)

def approval_termination(conversation: list) -> bool:
    if not conversation:
        return False
    return "APPROVE" in conversation[-1].text.upper()

workflow = GroupChatBuilder(
    participants=[coder, reviewer],
    termination_condition=approval_termination,
).build()

print("MAF Dev Team created: Coder + Reviewer")
```

Microsoft Agent Framework використовує **асинхронний API** — методи на кшталт `workflow.run()` є корутинами і потребують `await` або `async for`. У Jupyter Notebook `await` працює напряму на верхньому рівні, бо Jupyter має власний event loop. У звичайному Python-скрипті потрібно було б обгортати виклик у `asyncio.run()`.

```python
# ⚠️ REFERENCE ONLY — requires Azure credentials

# Run — agents exchange messages until Reviewer says APPROVE
async for event in workflow.run(
    "Write a Python class Stack with methods push, pop, peek, is_empty, size. "
    "Handle edge cases (pop from empty stack).",
    stream=True
):
    source = getattr(event, "source", "system")
    content = getattr(event, "content", "")
    if content:
        print("\n" + "=" * 50)
        print(f"{source}:")
        print(str(content)[:600])
```

---
# 🤖 Додаткові інструменти екосистеми

## Еволюція фреймворків: запит на надійність та легкість

Чому розробники шукають **додаткові інструменти** поза великими фреймворками?

| Проблема | Опис |
|----------|------|
| 🎭 **Промпт-інженерія під капотом** | Фреймворки ховають складні промпти, які важко відлагоджувати |
| 🏗️ **Надмірна архітектура** | Великий overhead для простих задач |
| 🔍 **Проблеми з прозорістю** | Ранні фреймворки (LangChain Chains, перший AutoGen) ускладнювали дебаг — важко зрозуміти, що саме пішло не так, коли помилка ховається за кількома шарами абстракцій |

**Тренд:** Легші, прозоріші фреймворки з мінімальним overhead. Smolagents та OpenAI Agents SDK — альтернативні підходи до оркестрації (code-driven та handoff відповідно).

## OpenAI Agents SDK — Production Handoffs

Еволюція підходу OpenAI до мультиагентності: Swarm (2024) був навчальним фреймворком для демонстрації патерну handoff, а в 2025 OpenAI випустила **Agents SDK** (`openai-agents`) — production-ready рішення з тією ж концепцією.

**Що додає Agents SDK поверх оригінальної ідеї Swarm:**
- **Tracing** — вбудований моніторинг кожного кроку агента
- **Guardrails** — валідація вводу/виводу для безпеки
- **Structured output** — гарантований формат відповіді
- **Sessions** — збереження контексту між викликами

**Головна ідея залишається — Handoff (передача естафети):**
- Agent — це набір інструкцій + tools + handoffs
- `handoffs=[agent_b]` — агент може передати управління іншому
- Немає центрального оркестратора — агенти передають управління один одному
- `Runner.run()` — запуск з автоматичним відстеженням handoffs

```
  ┌─────────┐  handoff  ┌──────────┐  handoff  ┌──────────┐
  │ Triage  │──────────▶│ Billing  │──────────▶│  Tech    │
  │  Agent  │           │  Agent   │           │  Agent   │
  └─────────┘           └──────────┘           └──────────┘
     handoffs=[billing, tech]
```

In [ ]:
from agents import Agent, Runner

# Specialist agents
billing = Agent(
    name="Billing",
    instructions="You are a billing agent. Answer questions about plans and payments.\n"
                 "Plans: Basic $5/mo, Standard $9/mo, Premium $14/mo.\nBe concise."
)

tech = Agent(
    name="Tech",
    instructions="You are a technical specialist. Help with internet and equipment issues.\n"
                 "First suggest restarting the router.\nBe concise."
)

# Triage agent with handoffs
triage = Agent(
    name="Triage",
    instructions="You are a router agent. Determine the type of customer request:\n"
                 "- Payments, plans, invoices -> hand off to Billing\n"
                 "- Technical issues (internet, Wi-Fi) -> hand off to Tech\n"
                 "Do not answer the request yourself.",
    handoffs=[billing, tech]
)

print("Agents SDK agents created: Triage -> Billing / Tech")

╭──────────────────────────────────────────────── Crew Completion ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Completed                                                                                       │
│  Name: crew                                                                                                     │
│  ID: 2181d99e-3408-46a4-93af-45601931df11                                                                       │
│  Final Output: Our plans are as follows:                                                                        │
│  - Basic plan: $5 per month                                                                                     │
│  - Standard plan: $9 per month                                                                                  │
│  - Premium plan: $14 per month                                                                                  │
│                                                                                                                 │
│  Regarding the setup fee, there is no specific mention of any setup charge, so it appears there is no setup     │
│  fee. If you need more details or have other questions, feel free to ask!                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── Tracing Status ─────────────────────────────────────────────────╮
│                                                                                                                 │
│  Info: Tracing is disabled.                                                                                     │
│                                                                                                                 │
│  To enable tracing, do any one of these:                                                                        │
│  • Set tracing=True in your Crew/Flow code                                                                      │
│  • Set CREWAI_TRACING_ENABLED=true in your project's .env file                                                  │
│  • Run: crewai traces enable                                                                                    │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

Agents SDK agents created: Triage -> Billing / Tech


In [ ]:
# Test automatic handoffs
test_messages = [
    "How much does the Premium plan cost?",
    "My internet has been down for an hour!",
]

for query in test_messages:
    print("\n" + "=" * 50)
    print(f"Customer: {query}")
    result = await Runner.run(triage, query)
    print(f"Routed to: {result.last_agent.name}")
    print(f"Response: {result.final_output}")


Customer: How much does the Premium plan cost?
Routed to: Billing
Response: The Premium plan costs $14 per month.

Customer: My internet has been down for an hour!
Routed to: Tech
Response: First, please try restarting your router by unplugging it for 30 seconds and plugging it back in. Let me know if the issue continues after that.


## Smolagents (Hugging Face) — Code-Driven Actions

**Альтернативний підхід:** Замість JSON для виклику інструментів, LLM генерує **звичайний Python-код**.

| Підхід | Традиційний (JSON) | Smolagents (Code) |
|--------|-------------------|------------------|
| **Формат** | `{"tool": "search", "query": "..."}` | `result = search("...")` |
| **Кілька інструментів** | Окремий виклик на кожен | Один блок коду з циклами |
| **Токени** | Більше (JSON overhead) | Менше (чистий Python) |
| **Гнучкість** | Обмежена схемою | Повна (if/for/функції) |

In [ ]:
from smolagents import CodeAgent, ToolCallingAgent, tool as smol_tool, OpenAIServerModel

smol_model = OpenAIServerModel(model_id="gpt-5.4-nano")

# Custom tools — smolagents uses @smol_tool (aliased to avoid shadowing CrewAI's @tool)
@smol_tool
def city_population(city: str) -> str:
    """Returns the population of a Ukrainian city.

    Args:
        city: City name (Kyiv, Kharkiv, Odesa, Lviv, Dnipro).
    """
    data = {
        "Kyiv": 3720000,
        "Kharkiv": 1200000,
        "Odesa": 1010000,
        "Lviv": 900000,
        "Dnipro": 931000
    }
    pop = data.get(city)
    if pop:
        return f"Population of {city}: {pop:,}"
    return f"No data for {city}. Available: {list(data.keys())}"

@smol_tool
def calculate_average(numbers: list) -> float:
    """Calculates the arithmetic mean of a list of numbers.

    Args:
        numbers: List of numbers to average.
    """
    return sum(numbers) / len(numbers)

print(f"Tools: {city_population.name}, {calculate_average.name}")
print(f"  Test: {city_population('Kyiv')}")

Tools: city_population, calculate_average
  Test: Population of Kyiv: 3,720,000


In [ ]:
# CodeAgent — generates Python code to solve the task
code_agent = CodeAgent(
    tools=[city_population, calculate_average],
    model=smol_model,
    additional_authorized_imports=["statistics"]
)

result = code_agent.run(
    "Find the population of Kyiv, Kharkiv, and Odesa. Calculate the average population of these three cities."
)
print(f"\nResult: {result}")

╭──────────────────────────────────────────────────── New run ────────────────────────────────────────────────────╮
│                                                                                                                 │
│ Find the population of Kyiv, Kharkiv, and Odesa. Calculate the average population of these three cities.        │
│                                                                                                                 │
╰─ OpenAIModel - gpt-5.4-nano ────────────────────────────────────────────────────────────────────────────────────╯

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ Step 1 ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

─ Executing parsed code: ──────────────────────────────────────────────────────────────────────────────────────── 
  cities = ["Kyiv", "Kharkiv", "Odesa"]                                                                            
  pop_strings = {}                                                                                                 
  for c in cities:                                                                                                 
      s = city_population(c)                                                                                       
      pop_strings[c] = s                                                                                           
      print(c, s)                                                                                                  
                                                                                                                   
  # Attempt to extract the first number from each string (population may include commas)                           
  def extract_number(text):                                                                                        
      import re                                                                                                    
      m = re.search(r'([0-9]+(?:,[0-9]+)*)', text)                                                                 
      if not m:                                                                                                    
          # fallback: find any float/int                                                                           
          m = re.search(r'([0-9]+(?:\.[0-9]+)?)', text)                                                            
      num_str = m.group(1).replace(',', '') if m else None                                                         
      return float(num_str)                                                                                        
                                                                                                                   
  nums = {c: extract_number(pop_strings[c]) for c in cities}                                                       
  print("Extracted numeric populations:", nums)                                                                    
                                                                                                                   
  from statistics import mean                                                                                      
  avg = mean(nums.values())                                                                                        
  print("Average:", avg)                                                                                           
                                                                                                                   
  final_answer({"populations": nums, "average_population": avg})                                                   
 ─────────────────────────────────────────────────────────────────────────────────────────────────────────────────

Execution logs:
Kyiv Population of Kyiv: 3,720,000
Kharkiv Population of Kharkiv: 1,200,000
Odesa Population of Odesa: 1,010,000
Extracted numeric populations: {'Kyiv': 3720000.0, 'Kharkiv': 1200000.0, 'Odesa': 1010000.0}
Average: 1976666.6666666667

Final answer: {'populations': {'Kyiv': 3720000.0, 'Kharkiv': 1200000.0, 'Odesa': 1010000.0}, 'average_population': 
1976666.6666666667}

[Step 1: Duration 2.74 seconds| Input tokens: 2,076 | Output tokens: 335]


Result: {'populations': {'Kyiv': 3720000.0, 'Kharkiv': 1200000.0, 'Odesa': 1010000.0}, 'average_population': 1976666.6666666667}


### 🔄 Smolagents: ToolCallingAgent vs CodeAgent

In [ ]:
# ToolCallingAgent — uses JSON tool calling (like traditional frameworks)
tool_agent = ToolCallingAgent(
    tools=[city_population],
    model=smol_model
)

result_tc = tool_agent.run("What is the population of Lviv?")
print(f"ToolCallingAgent: {result_tc}")

# CodeAgent — generates Python code (more efficient)
code_agent2 = CodeAgent(
    tools=[city_population],
    model=smol_model
)

result_ca = code_agent2.run("What is the population of Lviv?")
print(f"CodeAgent: {result_ca}")

print("\nBoth give the same result, but CodeAgent can combine tools in a single step.")

╭──────────────────────────────────────────────────── New run ────────────────────────────────────────────────────╮
│                                                                                                                 │
│ What is the population of Lviv?                                                                                 │
│                                                                                                                 │
╰─ OpenAIModel - gpt-5.4-nano ────────────────────────────────────────────────────────────────────────────────────╯

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ Step 1 ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

╭─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│ Calling tool: 'city_population' with arguments: {'city': 'Lviv'}                                                │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

Observations: Population of Lviv: 900,000

[Step 1: Duration 0.62 seconds| Input tokens: 1,032 | Output tokens: 18]

...

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ Step 1 ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

─ Executing parsed code: ──────────────────────────────────────────────────────────────────────────────────────── 
  pop = city_population("Lviv")                                                                                    
  final_answer(pop)                                                                                                
 ─────────────────────────────────────────────────────────────────────────────────────────────────────────────────

Final answer: Population of Lviv: 900,000

[Step 1: Duration 0.97 seconds| Input tokens: 2,024 | Output tokens: 87]

CodeAgent: Population of Lviv: 900,000

Both give the same result, but CodeAgent can combine tools in a single step.


### Порівняння: JSON-based vs Code-based агенти

```
  JSON-based (Традиційний підхід)      Code-based (Smolagents)

  Крок 1: {"tool": "search",           Крок 1 (один блок коду):
           "query": "Київ"}            ┌──────────────────────────┐
  → Результат                          │ results = []             │
  Крок 2: {"tool": "search",           │ for city in cities:      │
           "query": "Харків"}          │   r = search(city)       │
  → Результат                          │   results.append(r)      │
  Крок 3: {"tool": "search",           │ avg = sum(r) / len(r)    │
           "query": "Одеса"}           └──────────────────────────┘
  → Результат
  Крок 4: {"tool": "calculate",         Результат: 1 крок замість 4
           "expr": "avg(...)"}

  Значно більше токенів                 Значно менше токенів
  (окремий JSON на кожен виклик)        (один блок коду замінює
                                         кілька JSON-викликів)
```

## Інші гравці екосистеми

| Фреймворк / Сервіс | Фокус | Коли використовувати |
|---------------------|-------|---------------------|
| **LlamaIndex Workflows** | RAG + агенти | Система сильно зав'язана на роботу з документами |
| **AWS Bedrock Agents** | Cloud-native | Інфраструктура вже на AWS |
| **Google Vertex AI Agent Builder** | Cloud-native | Інфраструктура на GCP |

> 💡 Cloud-native рішення забирають оркестрацію на рівень хмарної інфраструктури,
> зменшуючи кількість коду, але збільшуючи vendor lock-in.

---
# ⚡ Як обрати правильний інструмент?

## Рекомендації за типом задачі

| Тип задачі | Рекомендований фреймворк | Чому |
|------------|-------------------------|------|
| 📝 **Створення контенту / Research** | CrewAI | Рольова модель ідеальна для творчих задач |
| 💻 **Генерація коду / Data Science** | MS Agent Framework або Smolagents | Нативне виконання коду |
| 🏢 **Складні бізнес-процеси (CRM, Support)** | LangGraph | Збереження стану, детермінований потік |
| 🧪 **Прототип / Навчання** | OpenAI Agents SDK | Мінімальний код, зрозумілий патерн handoff |

## Порівняльна таблиця фреймворків

| Характеристика | LangGraph | CrewAI | MS Agent Framework | OpenAI Agents SDK | Smolagents |
|----------------|-----------|--------|---------------------|-------------------|------------|
| **Парадигма** | Графова | Рольова | Розмовна | Handoff | Code-driven |
| **Контроль потоку** | ⭐⭐⭐⭐⭐ | ⭐⭐ | ⭐⭐⭐ | ⭐⭐⭐ | ⭐⭐⭐⭐ |
| **Простота старту** | ⭐⭐ | ⭐⭐⭐⭐⭐ | ⭐⭐⭐ | ⭐⭐⭐⭐⭐ | ⭐⭐⭐⭐ |
| **Збереження стану** | ✅ Вбудовано | ❌ | ⚠️ Часткове | ❌ | ❌ |
| **Виконання коду** | ❌ | ❌ | ✅ Docker | ❌ | ✅ Sandbox |
| **Екосистема** | LangChain | Власна | Microsoft | OpenAI | HuggingFace |
| **Production-ready** * | ✅ | ⚠️ | ⚠️ | ✅ | ⚠️ |
| **Ліцензія** | MIT | MIT | MIT | MIT | Apache 2.0 |

\* **✅** = стабільний API, широке production-використання; **⚠️** = API ще може змінюватись, менше battle-tested у enterprise, або обмежена персистентність стану.

---
# 📋 Підсумки

## Ключові висновки

| Фреймворк | Запам'ятайте головне |
|-----------|--------------------|
| **LangGraph** | Графи + State + Checkpoints = максимальний контроль |
| **CrewAI** | Agent(role, goal, backstory) + Crew = швидкий старт |
| **MS Agent Framework** | GroupChat + Code Execution = автономна розробка |
| **OpenAI Agents SDK** | Agent + handoffs + Runner = production handoffs |
| **Smolagents** | CodeAgent = менше токенів, більше гнучкості |

### Головний висновок

> **"Срібної кулі не існує."** Вибір фреймворку залежить від природи задачі:
> детермінована (LangGraph) чи творча (CrewAI), код (MAF/Smolagents) чи handoff (OpenAI Agents SDK).

Обирайте інструмент під задачу, а не задачу під інструмент.

## Ресурси та документація

| Фреймворк | Документація |
|-----------|-------------|
| LangGraph | https://langchain-ai.github.io/langgraph/ |
| CrewAI | https://docs.crewai.com/ |
| MS Agent Framework | https://learn.microsoft.com/en-us/azure/ai-services/agents/ |
| MS Agent Framework (SDK) | https://pypi.org/project/agent-framework/ |
| OpenAI Agents SDK | https://openai.github.io/openai-agents-python/ |
| Smolagents | https://huggingface.co/docs/smolagents/ |

**Додатково:**
- [Andrew Ng: Agentic Design Patterns](https://www.deeplearning.ai/the-batch/how-agents-can-improve-llm-performance/)
- [LangGraph: Multi-Agent Tutorials](https://langchain-ai.github.io/langgraph/tutorials/multi_agent/multi-agent-collaboration/)
- [AutoGen → Agent Framework Migration Guide](https://microsoft.github.io/autogen/)